In [ ]:
# %pip install langgraph   

In [ ]:
# %pip install langchain-openai

In [ ]:
# %pip install dotenv

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")
model.invoke("안녕하세요!")

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    """
    State 클래스는 TypedDict를 상속받습니다.
    
    속성:
        Messages (Annotated[list[str], add_messages]): 메시지들은 "list" 타입을 가집니다.
        'add_messages' 함수는 이 상태 키가 어떻게 업데이트되어야 하는지를 정의합니다.
        (이 경우, 메시지를 덮어쓰는 대신 리스트에 추가합니다)
    """
    messages: Annotated[list[str], add_messages]

# StateGraph 클래스를 사용하여 State 타입의 그래프 생성.
graph_builder = StateGraph(State)

In [ ]:
def generate(state: State):
    """
    주어진 상태를 기반으로 챗봇의 응답 메시지를 생성합니다.
    
    매개변수:
    state(State): 현재 대화 상태를 나타내는 객체로, 이전 메시지들이 포함되어 있습니다.
    
    반환값:
    dict: 모델이 생성한 응답 메시지를 포함하는 딕셔너리.
        형식은 {"messages: [응답 메시지]}입니다.
    """
    return {"messages": [model.invoke(state["messages"])]}

graph_builder.add_node("generate", generate)

In [ ]:
graph_builder.add_edge(START, "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

In [ ]:
# %pip install Ipython

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))

except Exception:
    pass

In [ ]:
response = graph.invoke({"messages" : ["안녕하세요! 저는 신호용입니다."]})

print(type(response))
response

In [ ]:
response["messages"].append("제 이름을 아시나요?")
graph.invoke(response)

In [ ]:
inputs = {"messages": [("human", '한국과 일본의 관계에 대해 자세히 알려줘.')]}
for chunk, _ in graph.stream(inputs, stream_mode="messages"):
    print(chunk.content, end="")